# Presupuesto, notas y hipótesis retiradas (P1-b.3c)
Tope de **210 000 tokens por pregunta** (se corta al 85 %), notas tuyas visibles para el agente en cada pregunta, y poder retirar las
hipótesis duplicadas. Ver `docs/presupuesto_y_notas.md`.

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from dfir_copilot.agent.graph import build_agent
from dfir_copilot.agent.llm import make_llm
from dfir_copilot.cases import CaseWorkspace

ws = CaseWorkspace.open("IDOR-INVOICES-2020Q4-TZ-SANTIAGO")
real = ws.engine(timeout_s=120)
engine, ps = ws.pseudonymized(timeout_s=120)
ledger = ws.ledger(real)
agent = build_agent(engine, ledger, make_llm(), max_steps=14)   # max_tokens=210_000 por defecto; añade max_tokens_case=… si quieres un tope del caso
print(agent.budget())

## 1. Cuánto se ha gastado ya en el caso
Sale del ledger: cada turno registra solo lo nuevo (una pregunta con aprobación no cuenta dos veces).

In [ ]:
turns = [{"seq": e["seq"], "estado": e["data"]["status"], "tokens_del_turno": e["data"].get("tokens_delta"),
          "cortado_por": e["data"].get("cut_by")} for e in ledger.entries("agent_turn")]
display(pd.DataFrame(turns))
print(agent.budget())

## 2. Dejar una nota que el agente verá en cada pregunta
`preview` enseña lo que se escribiría sin escribir nada. **Enumera los alias, no uses rangos.** Si escribes valores reales se traducen
solos; lo ambiguo se bloquea.

In [ ]:
texto = ("Aprobada solo en lo observable en los logs: cuatro cuentas (U-0032, U-0033, U-0034 y U-0035) con cinco IPs exclusivas y "
         "facturas que nadie más consulta. NO establece que las facturas sean ajenas (falta el mapa factura-propietario).")
print("Se escribiría:", agent.preview(texto).text)

In [ ]:
# agent.note(texto, status="inconclusive")   # descomenta para escribirla: el modelo la verá desde la próxima pregunta

## 3. Retirar las hipótesis duplicadas
Tienes hipótesis sobre el mismo fenómeno. Retirar no es una decisión sobre la evidencia: solo las descarta. Descomenta lo que quieras.

In [ ]:
display(pd.DataFrame([{"id": h["hypothesis_id"], "estado": h["status"], "con_criterio": bool(h.get("falsifier"))}
                      for h in agent.book.all()]))
# agent.retire("h-7a1d7576", "Reemplazada por h-aa79d3ab, con criterio de refutación e intentos", superseded_by="h-aa79d3ab")
# agent.retire("h-cfec051d", "Reemplazada por h-aa79d3ab", superseded_by="h-aa79d3ab")

## 4. Preguntar (gasta API)
`r.cut_by` dice si el presupuesto cortó la investigación.

In [ ]:
# r = agent.ask("Resume el estado del caso y propón la siguiente línea de investigación, teniendo en cuenta mis notas.")
# print(r.status, "| pasos:", r.steps, "| tokens:", r.tokens, "| cortada por:", r.cut_by)
# print(ps.reveal_any(r.answer) if r.answer else "(pausado: hay una hipótesis pendiente de tu aprobación)")
# print(agent.budget())

## 5. Replay copia por copia
Las consultas `DESCRIBE` que cambian porque la vista ganó una columna salen como `schema_drift`, no como alteración de datos.

In [ ]:
rep = ledger.replay([real, engine])
print(len(rep), "consultas |", sum(bool(x["match"]) for x in rep), "coinciden |", sum(bool(x.get("skipped")) for x in rep), "sin motor |",
      sum(bool(x.get("schema_drift")) for x in rep), "con deriva de esquema")
print({k: v for k, v in ledger.entries("replay")[-1]["data"].items() if k in ("mismatches", "skipped", "schema_drift")})